In [12]:
import sys
import numpy as np
import timeit
import statistics

sys.path.append('..')

from ruv.relative_utility_value import *
from ruv.damage_functions import *
from ruv.economic_models import *
from ruv.utility_functions import *
from ruv.helpers import *
from ruv.decision_methods import *
from util import *

In [23]:
awrc = '405209'
name = 'Acheron River at Tagerty'
area = 629.4

start_lt=1
end_lt=30

target_unity_risk_aversion = 0.3
max_damages = 10000
damages_quantile_threshold = 0.99
damages_shape = 0.2

# should take about an hour
parallel_nodes = 8
num_alphas = 20
benchmark_repeats = 15
num_timesteps = 1000

verbose = False

In [14]:
alphas = np.linspace(1/num_alphas, 1-1/num_alphas, num_alphas)
alphas

array([0.05      , 0.09736842, 0.14473684, 0.19210526, 0.23947368,
       0.28684211, 0.33421053, 0.38157895, 0.42894737, 0.47631579,
       0.52368421, 0.57105263, 0.61842105, 0.66578947, 0.71315789,
       0.76052632, 0.80789474, 0.85526316, 0.90263158, 0.95      ])

In [15]:

target_risk_premium = risk_aversion_coef_to_risk_premium(target_unity_risk_aversion, 1) 
adjusted_risk_aversion = risk_premium_to_risk_aversion_coef(target_risk_premium, max_damages)
adjusted_risk_aversion

c:\Users\me\research\relative-utility-value\laugesen_2023b\..\ruv\helpers.py:49: RuntimeWarning: overflow encountered in exp
  return np.log(0.5 * (np.exp(-A * gamble_size) + np.exp(A * gamble_size))) / (A * gamble_size) - risk_premium


3.0000000000003436e-05

In [16]:
obs, fcst, clim = load_data(awrc, start_lt, end_lt, area)
ref = clim

Loading data
	Loading data for 405209/muthre from lead-time 1 to 30
	obs shape: (8240,), fcst_ens shape: (8240, 100), clim shape: (8240, 508)


In [24]:
obs = obs[0:num_timesteps]
fcst = fcst[0:num_timesteps]
ref = ref[0:num_timesteps]

In [18]:
decision_definition = {
    'alphas': alphas,
    'target_unity_risk_aversion': target_unity_risk_aversion,
    'target_risk_premium': target_risk_premium,
    'adjusted_risk_aversion': adjusted_risk_aversion,
    'utility_function': [cara, {'A': adjusted_risk_aversion}],
    'economic_model': [cost_loss, cost_loss_analytical_spend],
    'decision_method': 'optimise_over_forecast_distribution',
    'decision_thresholds': None,
    'damage_function': [logistic, {'k': damages_shape, 'A': max_damages, 'threshold': np.nanquantile(obs, damages_quantile_threshold)}]
}

In [19]:
benchmark_results = timeit.repeat(
    lambda: relative_utility_value(obs, fcst, ref, decision_definition, parallel_nodes=parallel_nodes, verbose=verbose), 
    number=1, repeat=benchmark_repeats)
benchmark_results

[30.408149400000184,
 31.364584100001593,
 31.230395900000076,
 30.97977059999903,
 33.22251150000011,
 31.274790600000415,
 33.845933599999626,
 34.17910049999955,
 30.694723599999634,
 38.8897663000007,
 29.96607629999926,
 28.99139500000092,
 31.605352599999605,
 28.46233200000097,
 30.586801400000695]

In [20]:
times = np.array(benchmark_results) / num_alphas
times

array([1.52040747, 1.56822921, 1.5615198 , 1.54898853, 1.66112558,
       1.56373953, 1.69229668, 1.70895502, 1.53473618, 1.94448832,
       1.49830381, 1.44956975, 1.58026763, 1.4231166 , 1.52934007])

In [21]:
print(f"Mean: {statistics.mean(times)} seconds")
print(f"Median: {statistics.median(times)} seconds")
print(f"Standard Deviation: {statistics.stdev(times)} seconds")

Mean: 1.5856722780000079 seconds
Median: 1.5615197950000037 seconds
Standard Deviation: 0.1271562304818909 seconds


In [22]:
times_per_step = times/num_timesteps

print(f"Mean: {statistics.mean(times_per_step)} seconds")
print(f"Median: {statistics.median(times_per_step)} seconds")
print(f"Standard Deviation: {statistics.stdev(times_per_step)} seconds")

Mean: 0.001585672278000008 seconds
Median: 0.0015615197950000037 seconds
Standard Deviation: 0.0001271562304818909 seconds
